# Character-Level Text Generation with CNN, LSTM, RNN, and GRU Models

This notebook presents a complete character-level text generation project using multiple deep learning
architectures. The objective is to train neural networks that learn the statistical structure of text and
generate new sequences in the style of *Alice's Adventures in Wonderland*.

## Project Goals
- Build and train several sequence models:
  - **CharCNN** (1D convolutional model)
  - **LSTM** (baseline recurrent model)
  - **Modified LSTM** with variations in:
    - number of layers
    - hidden state dimensionality
    - dropout rates
    - other relevant hyperparameters
  - **RNN** and **GRU** alternatives for comparison

- Construct a character-level dataset using sliding windows of fixed length  
- Train all models using **cross-entropy loss**  
- Evaluate performance using:
  - **Character-level accuracy**
  - **Confusion matrices**
  - **Loss curves and training dynamics**

- Generate new text sequences from each trained model  
- Analyze and compare the generative behavior of CNN, LSTM, RNN, and GRU architectures  

## Key Learning Outcomes
- Understanding how different sequence models handle long-range dependencies  
- Observing the impact of architectural choices on prediction accuracy  
- Using confusion matrices to diagnose model errors  
- Exploring how hyperparameters influence generative quality  
- Gaining practical experience with character-level language modeling  

In [1]:
# Use GPU if available
import torch
train_on_gpu = torch.cuda.is_available()
if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')
device = torch.device("cuda:0" if train_on_gpu else "cpu")
print(device)

CUDA is not available.  Training on CPU ...
cpu


In [2]:
# mount with drive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [3]:
#change working directory
import os
os.chdir('/content/drive/MyDrive/cnn-lstm-textgen/')

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import re

# 1. Load dataset

We are going to use the text of the book "Alice's Adventures in Wonderland" to train our models. We need to download the text and prepare a dataset to load strings to train our models.


In [5]:
# Download the text file "Alice's Adventures in Wonderland" from Project Gutenberg
!wget -O wonderland.txt https://www.gutenberg.org/ebooks/11.txt.utf-8

# Load the text data from the downloaded file
filename = "wonderland.txt"
with open(filename, 'r', encoding='utf-8') as f:
  raw_text = f.read()

# Convert all characters to lowercase
raw_text = raw_text.lower()

# Remove non-alphanumeric characters
raw_text = re.sub(r'\n', ' ', raw_text)
raw_text = re.sub(r'[^A-Za-z ]+', '', raw_text)

print(raw_text[500:700])
# Create a set of unique characters in the text
unique_chars = set(raw_text)

# Sort the unique characters
chars = sorted(list(unique_chars))

# Create a dictionary mapping each unique character to a unique integer
char_to_int = dict((c, i) for i, c in enumerate(chars))

# Split the text into training and testing sets
train_start = int(len(raw_text) * 0.1)  # Starting index for training set (10% of text)
train_end = int(len(raw_text) * 0.8)  # Ending index for training set (80% of text)
test_start = train_end  # Starting index for testing set (remaining 20% of text)

raw_text_train = raw_text[train_start:train_end]  # Extract training text
raw_text_test = raw_text[test_start:]  # Extract testing text

# Calculate and print some summary statistics
n_chars_train = len(raw_text_train)
n_chars_test = len(raw_text_test)
n_vocab = len(chars)

print("Total Characters train:", n_chars_train)
print("Total Characters test:", n_chars_test)
print("Total Unique Characters (Vocabulary Size):", n_vocab)
print("Chars: ",chars)

--2026-05-07 13:21:19--  https://www.gutenberg.org/ebooks/11.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/11/pg11.txt [following]
--2026-05-07 13:21:20--  http://www.gutenberg.org/cache/epub/11/pg11.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.gutenberg.org/cache/epub/11/pg11.txt [following]
--2026-05-07 13:21:20--  https://www.gutenberg.org/cache/epub/11/pg11.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 174314 (170K) [text/plain]
Saving to: ‘wonderland.txt’

wonderland.txt      100%[===================>] 170.23K  --.-KB/s    in 0.1s   

Prepare the dataset of input to output pairs encoded as integers.
We need to:
1. Convert string of characters to a list of charachter indexes (using the chat_to_ind dictionary previouslty created)
2. Store the list of indexes and the targets in a collection of sequences
3. Prepare the Pytorch Tensors

The model takes input sequences of 100 characters and predicts the character 101

In [6]:
# Define the sequence length for character prediction
seq_length = 100

# List to store sequences of characters as integer indices
dataX = []

# List to store target characters (next character) as integer indices
dataY = []

# Loop through the training text with a stride of 1 character
for i in range(0, n_chars_train - seq_length, 1):
  # Extract an input sequence of length 'seq_length' from the training text
  seq_in = raw_text_train[i:i + seq_length]

  # Extract the target character (next character to predict)
  seq_out = raw_text_train[i + seq_length]

  # Convert characters in the input sequence to their integer indices using the mapping dictionary
  dataX.append([char_to_int[char] for char in seq_in])

  # Append the target character's integer index to the output list
  dataY.append(char_to_int[seq_out])

# Count the total number of training patterns (sequences-target character pairs)
n_patterns = len(dataX)
print("Total Patterns train: ", n_patterns)

# Convert the lists of integer indices (dataX and dataY) into PyTorch tensors
X_train = torch.tensor(dataX, dtype=torch.float32)
y_train = torch.tensor(dataY)

# Reshape the input sequences into a 3D tensor with dimensions:
#   - n_patterns: Number of training patterns
#   - seq_length: Length of the input sequence
#   - 1: Number of features (one-hot encoded characters can be represented as floats here)
X_train = X_train.reshape(n_patterns, seq_length, 1)

# Normalize the input sequences by dividing each element by the vocabulary size
# This helps the training process of the model
X_train = X_train / float(n_vocab)

# Prepare testing data (similar to training data preparation)

y_train = torch.tensor(dataY)

#test
dataX = []
dataY = []
for i in range(0, n_chars_test - seq_length, 1):
    seq_in = raw_text_test[i:i + seq_length]
    seq_out = raw_text_test[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])
n_patterns = len(dataX)
print("Total Patterns test: ", n_patterns)
# Convert the testing data lists to tensors, reshape, normalize, and create target tensor
X_test = torch.tensor(dataX, dtype=torch.float32).reshape(n_patterns, seq_length, 1)
X_test = X_test / float(n_vocab)
y_test = torch.tensor(dataY)

Total Patterns train:  108238
Total Patterns test:  30854


# 2. CNN definition

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class CharCNN(nn.Module):

    def __init__(self, n_vocab):
        super(CharCNN, self).__init__()

        self.conv1 = nn.Conv1d(1, 8, 5)  # First convolutional layer
        self.pool = nn.MaxPool1d(2, 2)  # Max pooling layer
        self.conv2 = nn.Conv1d(8, 16, 5)  # Second convolutional layer
        self.fc1 = nn.Linear(16 * 22, 128)  # First fully-connected layer
        self.fc2 = nn.Linear(128, 64)  # Second fully-connected layer
        self.fc3 = nn.Linear(64, n_vocab)  # Output layer

    def forward(self, x):
        """
        Defines the forward pass of the neural network.

        Args:
            x (torch.Tensor): Input tensor representing the images.

        Returns:
            torch.Tensor: Output tensor representing the class probabilities.
        """
        x = self.pool(F.relu(self.conv1(x)))  # First convolutional layer with ReLU activation and pooling
        x = self.pool(F.relu(self.conv2(x)))  # Second convolutional layer with ReLU activation and pooling
        # print(x.shape)
        x = x.view(x.shape[0],-1)  # Flatten the output from convolutional layers
        # print(x.shape)
        x = F.relu(self.fc1(x))  # First fully-connected layer with ReLU activation
        x = F.relu(self.fc2(x))  # Second fully-connected layer with ReLU activation
        x = self.fc3(x)  # Output layer
        return x

net = CharCNN(n_vocab)

# Create a random input tensor (batch size 128, sequence length 100, feature size 1)
inp = torch.randn(128, 1, 100)

# Run the model with the input tensor and get the output
out = net(inp)

print(out.shape)

torch.Size([128, 27])


# 3. LSTM Model definition.

In [8]:
class CharModel(nn.Module):
  """
  Character-Level Language Model using LSTM.

  This class defines a recurrent neural network (RNN) model with a Long Short-Term Memory (LSTM) layer
  for character-level language modeling.

  Attributes:
      lstm: An LSTM layer with specified input size, hidden size, number of layers, and dropout rate.
      linear: A linear layer to map the LSTM output to the vocabulary size (number of characters).
  """

  def __init__(self, n_vocab):
    """
    Initializes the model with an LSTM layer and a linear layer.

    Args:
        None
    """
    super().__init__()  # Call the superclass constructor

    # Define the LSTM layer
    self.lstm = nn.LSTM(
        input_size=1,  # Input size: one-hot encoded characters (represented as floats here)
        hidden_size=256,  # Hidden size of the LSTM layer
        num_layers=2,  # Number of LSTM layers stacked on top of each other
        batch_first=True,  # Input tensors are of shape (batch_size, seq_len, features)
        dropout=0.2  # Dropout rate for regularization
    )

    # Define the linear layer for output
    self.linear = nn.Linear(256, n_vocab)  # Map LSTM output to vocabulary size (number of characters)

  def forward(self, x):
    """
    Defines the forward pass of the model.

    Args:
        x: A PyTorch tensor of shape (batch_size, seq_len, features) representing the input sequences.

    Returns:
        A PyTorch tensor of shape (batch_size, vocab_size) containing the model's output logits.
    """

    # Pass the input sequence through the LSTM layer
    # The output (`x`) will be a tuple containing the output tensor and the hidden/cell states (not used here)
    x, _ = self.lstm(x)

    # Print the output shape for debugging purposes (can be commented out)
    # print(x.shape)

    # Take only the last output from the sequence (represents the model's prediction based on the entire sequence)
    x = x[:, -1, :]  # Select the last element from the sequence dimension

    # Pass the LSTM output through the linear layer to get logits for the next character prediction
    x = self.linear(x)

    return x